### Explicación del Código

El código en `validacion_modelo_NER.ipynb` utiliza un modelo NER entrenado con spaCy para procesar textos y generar una visualización interactiva en formato HTML. Realiza las siguientes tareas:

1. **Carga del Modelo NER**:
   - Carga el modelo spaCy desde la carpeta `./model-best`.

2. **Lectura de Datos**:
   - Lee un archivo CSV ubicado en `/home/sotavento/Documents/tejer_red/original_data/repd_vp_cedulas_principal.csv`.
   - Selecciona una muestra aleatoria de textos de la columna `descripcion_desaparicion`.

3. **Anotación de Textos**:
   - Procesa los textos seleccionados con el modelo NER.
   - Resalta las entidades detectadas (e.g., `NOMBRE`, `DOMICILIO`, `FECHA`) con colores específicos definidos en `ENTITY_COLORS`.

4. **Generación de HTML**:
   - Crea un archivo HTML interactivo (`ner_results.html`) que muestra las entidades detectadas en cada texto.
   - Incluye estilos personalizados y estructura visual con Bootstrap.

5. **Archivos y Carpetas Relevantes**:
   - **Modelo NER**: `./model-best`.
   - **Entrada**: `/home/sotavento/Documents/tejer_red/original_data/repd_vp_cedulas_principal.csv`.
   - **Salida**: `./ner_results.html`.

In [ ]:
# Instalar dependencias necesarias
!pip install spacy pandas

In [ ]:
import spacy
import pandas as pd
import random
import os
from datetime import datetime
import html

# Definir colores para cada clase de entidad
ENTITY_COLORS = {
    "NOMBRE": "#D5CCFF",     # Lila claro
    "DOMICILIO": "#FFCCCC",  # Rosa claro
    "FECHA": "#CCE5FF",      # Azul claro
    "HORA": "#FFFFCC",       # Amarillo claro
    "TELEFONO": "#CCFFCC",   # Verde claro
    "EXP": "#FFE5CC",        # Naranja claro
    "PLACA": "#E5CCFF"       # Púrpura claro
}

def generate_html_header():
    """Genera el encabezado HTML con estilos y estructura básica"""
    return """<!DOCTYPE html>
    <html lang="es">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Visualizador de Resultados NER</title>
        <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0-alpha3/dist/css/bootstrap.min.css" rel="stylesheet">
        <style>
            body { font-family: Arial, sans-serif; line-height: 1.6; padding: 20px; }
            .entity { display: inline-block; padding: 2px 4px; margin: 1px; border-radius: 4px; font-size: 0.9em; color: #000; }
            .entity small { font-size: 0.75em; color: #555; }
            .annotation-container { margin-bottom: 20px; padding: 15px; border: 1px solid #ddd; border-radius: 5px; background-color: #f9f9f9; }
            .stats-box { background-color: #e9ecef; padding: 15px; margin-bottom: 20px; border-radius: 5px; }
        </style>
    </head>
    <body>
    <h1>Resultados del Modelo NER</h1>
    """

def generate_html_footer():
    """Genera el pie de página HTML"""
    return """
    </body>
    </html>
    """

def annotate_text(text, doc):
    """Genera anotaciones HTML para un texto procesado"""
    annotated_text = ""
    last_end = 0
    for ent in doc.ents:
        annotated_text += html.escape(text[last_end:ent.start_char])
        color = ENTITY_COLORS.get(ent.label_, "#FFFFFF")
        annotated_text += f'<span class="entity" style="background-color: {color};">{html.escape(ent.text)} <small>({ent.label_})</small></span>'
        last_end = ent.end_char
    annotated_text += html.escape(text[last_end:])
    return annotated_text

def process_and_generate_html(model_path, csv_path, output_html_path, n_samples=1):
    """Procesa muestras aleatorias y genera un archivo HTML con los resultados"""
    print(f"Loading model from {model_path}...")
    nlp = spacy.load(model_path)

    try:
        print(f"Reading descriptions from CSV...")
        df = pd.read_csv(csv_path, usecols=['descripcion_desaparicion'])
        samples = df['descripcion_desaparicion'].sample(n=n_samples)

        print(f"Generating HTML visualization...")
        with open(output_html_path, "w", encoding="utf-8") as html_file:
            html_file.write(generate_html_header())

            for i, text in enumerate(samples, 1):
                doc = nlp(text)
                annotated_text = annotate_text(text, doc)
                html_file.write(f'<div class="annotation-container"><h2>Muestra {i}</h2><p>{annotated_text}</p></div>')

            html_file.write(generate_html_footer())

        print(f"HTML file generated at: {output_html_path}")

    except Exception as e:
        print(f"Error: {str(e)}")

if __name__ == "__main__":
    model_path = "./model-best"
    csv_path = "/home/sotavento/Documents/tejer_red/original_data/repd_vp_cedulas_principal.csv"
    
    # Generar un timestamp con la fecha y hora actual
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_html_path = f"./ner_results.html"

    process_and_generate_html(model_path, csv_path, output_html_path)

In [ ]:
import spacy
import pandas as pd
import os
from datetime import datetime
import html
import json

# Definir colores para cada clase de entidad
ENTITY_COLORS = {
    "NOMBRE": "#D5CCFF",     # Lila claro
    "DOMICILIO": "#FFCCCC",  # Rosa claro
    "FECHA": "#CCE5FF",      # Azul claro
    "HORA": "#FFFFCC",       # Amarillo claro
    "TELEFONO": "#CCFFCC",   # Verde claro
    "EXP": "#FFE5CC",        # Naranja claro
    "PLACA": "#E5CCFF"       # Púrpura claro
}

def generate_html_header():
    """Genera el encabezado HTML con estilos y estructura básica"""
    return """<!DOCTYPE html>
    <html lang="es">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Visualizador de Resultados NER</title>
        <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0-alpha3/dist/css/bootstrap.min.css" rel="stylesheet">
        <style>
            body { font-family: Arial, sans-serif; line-height: 1.6; padding: 20px; }
            .entity { display: inline-block; padding: 2px 4px; margin: 1px; border-radius: 4px; font-size: 0.9em; color: #000; }
            .entity small { font-size: 0.75em; color: #555; }
            .annotation-container { margin-bottom: 20px; padding: 15px; border: 1px solid #ddd; border-radius: 5px; background-color: #f9f9f9; }
            .stats-box { background-color: #e9ecef; padding: 15px; margin-bottom: 20px; border-radius: 5px; }
        </style>
    </head>
    <body>
    <h1>Resultados del Modelo NER</h1>
    """

def generate_html_footer():
    """Genera el pie de página HTML"""
    return """
    </body>
    </html>
    """

def annotate_text(text, doc):
    """Genera anotaciones HTML para un texto procesado"""
    annotated_text = ""
    last_end = 0
    for ent in doc.ents:
        annotated_text += html.escape(text[last_end:ent.start_char])
        color = ENTITY_COLORS.get(ent.label_, "#FFFFFF")
        annotated_text += f'<span class="entity" style="background-color: {color};">{html.escape(ent.text)} <small>({ent.label_})</small></span>'
        last_end = ent.end_char
    annotated_text += html.escape(text[last_end:])
    return annotated_text

def process_and_generate_outputs(model_path, csv_path, output_html_path, output_json_path, n_samples=2):
    """Procesa muestras aleatorias y genera archivos HTML y JSON con los resultados"""
    print(f"Loading model from {model_path}...")
    nlp = spacy.load(model_path)

    try:
        print(f"Reading descriptions from CSV...")
        df = pd.read_csv(csv_path, usecols=['descripcion_desaparicion'])
        samples = df['descripcion_desaparicion'].sample(n=n_samples)

        annotations = []

        print(f"Generating HTML and JSON outputs...")
        with open(output_html_path, "w", encoding="utf-8") as html_file:
            html_file.write(generate_html_header())

            for i, text in enumerate(samples, 1):
                doc = nlp(text)
                annotated_text = annotate_text(text, doc)

                # Escribir en el archivo HTML
                html_file.write(f'<div class="annotation-container"><h2>Muestra {i}</h2><p>{annotated_text}</p></div>')

                # Generar anotaciones para el archivo JSON
                entities = []
                for ent in doc.ents:
                    entities.append({
                        "start": ent.start_char,
                        "end": ent.end_char,
                        "label": ent.label_,
                        "text": ent.text
                    })
                annotations.append({
                    "id": i,
                    "text": text,
                    "entities": entities
                })

            html_file.write(generate_html_footer())

        # Guardar las anotaciones en un archivo JSON
        with open(output_json_path, "w", encoding="utf-8") as json_file:
            json.dump(annotations, json_file, indent=4, ensure_ascii=False)

        print(f"HTML file generated at: {output_html_path}")
        print(f"JSON file generated at: {output_json_path}")

    except Exception as e:
        print(f"Error: {str(e)}")

if __name__ == "__main__":
    model_path = "./model-best"
    csv_path = "/home/sotavento/Documents/tejer_red/original_data/repd_vp_cedulas_principal.csv"
    
    # Generar un timestamp con la fecha y hora actual
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_html_path = f"./ner_results.html"
    output_json_path = f"./ner_results.json"

    process_and_generate_outputs(model_path, csv_path, output_html_path, output_json_path)

In [ ]:
def call_deepseek(text, entities):
    """
    Envía un texto y las entidades detectadas a la API de DeepSeek para validación.

    Args:
        text (str): El texto a analizar.
        entities (list): Lista de entidades detectadas con sus posiciones y etiquetas.

    Returns:
        list: Lista de entidades validadas por DeepSeek.
    """
    try:
        # Construir el mensaje estructurado para DeepSeek
        entity_info = "\n".join(
            [f"- {entity['label']}: '{entity['text']}' (start: {entity['start']}, end: {entity['end']})"
             for entity in entities]
        )
        class_descriptions = "\n".join(
            [f"{cls}: {desc}" for cls, desc in DESCRIPTIONS.items()]
        )
        prompt = [
            {"role": "system", "content": "You are a helpful assistant that validates entities in text."},
            {"role": "user", "content": f"Here is a text:\n\n{text}\n\nAnd here are the detected entities:\n{entity_info}\n\nThese are the predefined classes and their descriptions:\n{class_descriptions}\n\nPlease validate these entities and provide corrections if necessary."},
        ]

        # Realizar la solicitud a DeepSeek utilizando el cliente OpenAI
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=prompt,
            stream=False
        )

        # Extraer el contenido del mensaje
        content = response.choices[0].message.content

        # Imprimir la respuesta de DeepSeek en la consola
        print("\n--- Respuesta de DeepSeek ---")
        print(content)
        print("\n-----------------------------")

        # Procesar manualmente las entidades desde el texto devuelto
        validated_entities = []
        lines = content.split("\n")
        for line in lines:
            if "**" in line and "(start:" in line and "(end:" in line:  # Buscar líneas con formato de entidad
                parts = line.split(":")
                label = parts[0].strip("**").strip()
                text_part = parts[1].split("(start:")[0].strip().strip("'")
                start = int(parts[1].split("(start:")[1].split(",")[0].strip())
                end = int(parts[1].split("(end:")[1].strip(")").strip())
                validated_entities.append({"label": label, "text": text_part, "start": start, "end": end})

        return validated_entities

    except Exception as e:
        print(f"Error al llamar a la API de DeepSeek: {e}")
        return []

def calculate_case_metrics(predicted_entities, validated_entities):
    """
    Calcula métricas por caso y clasifica entidades como correctas o incorrectas.

    Args:
        predicted_entities (list): Entidades predichas por el modelo.
        validated_entities (list): Entidades validadas por DeepSeek.

    Returns:
        tuple: Listas de entidades correctas e incorrectas.
    """
    correct = []
    incorrect = []

    for pred in predicted_entities:
        if pred in validated_entities:
            correct.append(pred)
        else:
            incorrect.append(pred)

    return correct, incorrect

def calculate_metrics(y_true, y_pred):
    """
    Calcula métricas como precisión, recall y F1-score.

    Args:
        y_true (list): Etiquetas verdaderas.
        y_pred (list): Etiquetas predichas.

    Returns:
        dict: Diccionario con las métricas calculadas.
    """
    if not y_true or not y_pred:
        print("Advertencia: y_true o y_pred están vacíos. No se pueden calcular métricas.")
        return {
            "classification_report": {},
            "precision": 0.0,
            "recall": 0.0,
            "f1_score": 0.0
        }

    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    return {
        "classification_report": report,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

if __name__ == "__main__":
    # Ruta al archivo JSON generado previamente
    json_path = "/home/sotavento/Documents/tejer_red/NER/3_prueba_modelo/ner_results.json"
    output_html_path = "./ner_validation_report.html"

    # Cargar datos desde el JSON
    def load_json(json_path):
        try:
            with open(json_path, "r", encoding="utf-8") as file:
                return json.load(file)
        except Exception as e:
            print(f"Error al cargar el archivo JSON: {e}")
            return []

    data = load_json(json_path)

    cases = []
    y_true = []
    y_pred = []

    # Procesar cada texto en el JSON
    for entry in data:
        text = entry["text"]
        predicted_entities = entry["entities"]

        # Obtener entidades validadas por DeepSeek
        validated_entities = call_deepseek(text, predicted_entities)

        # Calcular métricas por caso
        correct, incorrect = calculate_case_metrics(predicted_entities, validated_entities)

        # Agregar resultados del caso
        cases.append({
            "id": entry["id"],
            "text": text,
            "predicted": predicted_entities,
            "validated": validated_entities,
            "correct": correct,
            "incorrect": incorrect
        })

        # Agregar etiquetas para métricas generales
        y_true.extend([entity["label"] for entity in validated_entities])
        y_pred.extend([entity["label"] for entity in predicted_entities])

    # Calcular métricas generales
    metrics = calculate_metrics(y_true, y_pred)

    # Generar reporte HTML
    generate_html_report(cases, metrics, output_html_path)

## Anonimizacion

In [ ]:
import spacy
import pandas as pd
import os
from datetime import datetime
import html

# Definir colores para cada clase de entidad
ENTITY_COLORS = {
    "NOMBRE": "#D5CCFF",     # Lila claro
    "DOMICILIO": "#FFCCCC",  # Rosa claro
    "FECHA": "#CCE5FF",      # Azul claro
    "HORA": "#FFFFCC",       # Amarillo claro
    "TELEFONO": "#CCFFCC",   # Verde claro
    "EXP": "#FFE5CC",        # Naranja claro
    "PLACA": "#E5CCFF"       # Púrpura claro
}

# Entidades a anonimizar
ENTIDADES_A_ANONIMIZAR = ["NOMBRE", "DOMICILIO"]

def anonymize_text(text, doc):
    """
    Anonimiza el texto reemplazando las entidades especificadas con su tipo.

    Args:
        text (str): Texto original.
        doc (spacy.tokens.Doc): Documento procesado por el modelo NER.

    Returns:
        str: Texto anonimizado.
    """
    anonymized_text = ""
    last_end = 0

    for ent in doc.ents:
        anonymized_text += text[last_end:ent.start_char]
        if ent.label_ in ENTIDADES_A_ANONIMIZAR:
            anonymized_text += f"[{ent.label_}]"
        else:
            anonymized_text += ent.text
        last_end = ent.end_char

    anonymized_text += text[last_end:]
    return anonymized_text

def generate_html_anonymized_column(anonymized_column, output_html_path):
    """
    Genera un archivo HTML para visualizar la columna anonimizada con colores por entidad.

    Args:
        anonymized_column (pd.Series): Columna anonimizada.
        output_html_path (str): Ruta del archivo HTML de salida.
    """
    html_header = """<!DOCTYPE html>
    <html lang="es">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Visualización de Columna Anonimizada</title>
        <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0-alpha3/dist/css/bootstrap.min.css" rel="stylesheet">
        <style>
            body { font-family: Arial, sans-serif; line-height: 1.6; padding: 20px; }
            .entity { display: inline-block; padding: 2px 4px; margin: 1px; border-radius: 4px; font-size: 0.9em; color: #000; }
            .entity small { font-size: 0.75em; color: #555; }
            .container { max-width: 800px; margin: auto; }
            ul { padding-left: 20px; }
        </style>
    </head>
    <body>
    <div class="container">
        <h1 class="mt-4">Visualización de Columna Anonimizada</h1>
        <ul class="list-group">
    """
    html_footer = """
        </ul>
    </div>
    </body>
    </html>
    """

    def colorize_text(text):
        """
        Aplica colores a las etiquetas de las entidades en el texto anonimizado.

        Args:
            text (str): Texto anonimizado.

        Returns:
            str: Texto con etiquetas coloreadas.
        """
        for entity, color in ENTITY_COLORS.items():
            text = text.replace(f"[{entity}]", f'<span class="entity" style="background-color: {color};">[{entity}]</span>')
        return text

    html_body = ""
    for i, text in enumerate(anonymized_column, 1):
        colored_text = colorize_text(html.escape(text))
        html_body += f"<li class='list-group-item'>{colored_text}</li>"

    with open(output_html_path, "w", encoding="utf-8") as html_file:
        html_file.write(html_header + html_body + html_footer)

    print(f"HTML file generated at: {output_html_path}")

def process_and_anonymize_column(model_path, csv_path, column_name, output_csv_path, output_html_path):
    """
    Procesa un archivo CSV, anonimiza una columna específica y genera un archivo CSV y HTML.

    Args:
        model_path (str): Ruta al modelo NER.
        csv_path (str): Ruta al archivo CSV de entrada.
        column_name (str): Nombre de la columna a anonimizar.
        output_csv_path (str): Ruta del archivo CSV de salida.
        output_html_path (str): Ruta del archivo HTML de salida.
    """
    print(f"Loading model from {model_path}...")
    nlp = spacy.load(model_path)

    try:
        print(f"Reading CSV from {csv_path}...")
        df = pd.read_csv(csv_path)

        if column_name not in df.columns:
            raise ValueError(f"La columna '{column_name}' no existe en el archivo CSV.")

        print(f"Processing column '{column_name}' for anonymization...")
        anonymized_column = []

        for text in df[column_name]:
            doc = nlp(str(text))
            anonymized_text = anonymize_text(text, doc)
            anonymized_column.append(anonymized_text)

        # Agregar la columna anonimizada al DataFrame
        df[f"{column_name}_anonimizada"] = anonymized_column

        # Guardar el nuevo CSV
        df.to_csv(output_csv_path, index=False)
        print(f"CSV file with anonymized column generated at: {output_csv_path}")

        # Generar el archivo HTML para la columna anonimizada
        generate_html_anonymized_column(anonymized_column, output_html_path)

    except Exception as e:
        print(f"Error: {str(e)}")

if __name__ == "__main__":
    model_path = "./model-best"
    csv_path = "/home/sotavento/Documents/tejer_red/original_data/cedulassample.csv"
    column_name = "descripcion_desaparicion"  # Nombre de la columna a anonimizar
    output_csv_path = "./anonimized_output.csv"
    output_html_path = "./anonimized_column.html"

    process_and_anonymize_column(model_path, csv_path, column_name, output_csv_path, output_html_path)

# Anonimizacion + capas

In [ ]:
import spacy
import pandas as pd
import re
from datetime import datetime
import html

# Definir colores para cada clase de entidad y reglas regex
ENTITY_COLORS = {
    "NOMBRE": "#D5CCFF",     # Lila claro
    "DOMICILIO": "#FFCCCC",  # Rosa claro
    "FECHA": "#CCE5FF",      # Azul claro
    "HORA": "#FFFFCC",       # Amarillo claro
    "TELEFONO": "#CCFFCC",   # Verde claro
    "EXP": "#FFE5CC",        # Naranja claro
    "PLACA": "#E5CCFF",      # Púrpura claro
    "REGEX": "#FFD700"       # Dorado para regex
}

# Entidades a anonimizar
ENTIDADES_A_ANONIMIZAR = ["NOMBRE", "DOMICILIO"]

# Entidades protegidas (no afectadas por regex)
ENTIDADES_PROTEGIDAS = ["FECHA", "HORA"]

def anonymize_text_with_regex(text, doc):
    """
    Anonimiza el texto utilizando NER y reglas regex, respetando las entidades protegidas.

    Args:
        text (str): Texto original.
        doc (spacy.tokens.Doc): Documento procesado por el modelo NER.

    Returns:
        str: Texto anonimizado.
    """
    anonymized_text = ""
    last_end = 0

    # Crear un mapa de entidades protegidas para evitar que regex las afecte
    protected_entities = []
    for ent in doc.ents:
        if ent.label_ in ENTIDADES_PROTEGIDAS:
            protected_entities.append((ent.start_char, ent.end_char))

    # Aplicar anonimización basada en NER
    for ent in doc.ents:
        anonymized_text += text[last_end:ent.start_char]
        if ent.label_ in ENTIDADES_A_ANONIMIZAR:
            anonymized_text += f"[{ent.label_}]"
        else:
            anonymized_text += f'<span style="background-color: {ENTITY_COLORS[ent.label_]};">{ent.text}</span>'
        last_end = ent.end_char

    anonymized_text += text[last_end:]

    # Aplicar reglas regex globales respetando las entidades protegidas
    def regex_replace(match):
        start, end = match.span()
        # Verificar si el match está dentro de una entidad protegida
        for protected_start, protected_end in protected_entities:
            if protected_start <= start < protected_end or protected_start < end <= protected_end:
                return match.group()  # Devolver el texto original si está protegido
        return f'<span style="background-color: {ENTITY_COLORS["REGEX"]};">X</span>'  # Resaltar con color dorado

    # Regex para eliminar todos los dígitos (excepto en entidades protegidas)
    regex_pattern = r"\d+"
    matches = list(re.finditer(regex_pattern, anonymized_text))
    for match in matches:
        start, end = match.span()
        # Verificar si el match está dentro de una entidad protegida
        if any(protected_start <= start < protected_end or protected_start < end <= protected_end
               for protected_start, protected_end in protected_entities):
            continue  # Saltar si está protegido
        anonymized_text = anonymized_text[:start] + regex_replace(match) + anonymized_text[end:]

    return anonymized_text

def generate_html_anonymized_column(anonymized_column, output_html_path):
    """
    Genera un archivo HTML para visualizar la columna anonimizada con colores por entidad.

    Args:
        anonymized_column (pd.Series): Columna anonimizada.
        output_html_path (str): Ruta del archivo HTML de salida.
    """
    html_header = """<!DOCTYPE html>
    <html lang="es">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Visualización de Columna Anonimizada</title>
        <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0-alpha3/dist/css/bootstrap.min.css" rel="stylesheet">
        <style>
            body { font-family: Arial, sans-serif; line-height: 1.6; padding: 20px; }
            .entity { display: inline-block; padding: 2px 4px; margin: 1px; border-radius: 4px; font-size: 0.9em; color: #000; }
            .entity small { font-size: 0.75em; color: #555; }
            .container { max-width: 800px; margin: auto; }
            ul { padding-left: 20px; }
        </style>
    </head>
    <body>
    <div class="container">
        <h1 class="mt-4">Visualización de Columna Anonimizada</h1>
        <ul class="list-group">
    """
    html_footer = """
        </ul>
    </div>
    </body>
    </html>
    """

    def colorize_text(text):
        """
        Aplica colores a las etiquetas de las entidades en el texto anonimizado.

        Args:
            text (str): Texto anonimizado.

        Returns:
            str: Texto con etiquetas coloreadas.
        """
        for entity, color in ENTITY_COLORS.items():
            text = text.replace(f"[{entity}]", f'<span class="entity" style="background-color: {color};">[{entity}]</span>')
        return text

    html_body = ""
    for i, text in enumerate(anonymized_column, 1):
        colored_text = colorize_text(html.unescape(text))
        html_body += f"<li class='list-group-item'>{colored_text}</li>"

    with open(output_html_path, "w", encoding="utf-8") as html_file:
        html_file.write(html_header + html_body + html_footer)

    print(f"HTML file generated at: {output_html_path}")

def process_and_anonymize_column(model_path, csv_path, column_name, output_csv_path, output_html_path):
    """
    Procesa un archivo CSV, anonimiza una columna específica y genera un archivo CSV y HTML.

    Args:
        model_path (str): Ruta al modelo NER.
        csv_path (str): Ruta al archivo CSV de entrada.
        column_name (str): Nombre de la columna a anonimizar.
        output_csv_path (str): Ruta del archivo CSV de salida.
        output_html_path (str): Ruta del archivo HTML de salida.
    """
    print(f"Loading model from {model_path}...")
    nlp = spacy.load(model_path)

    try:
        print(f"Reading CSV from {csv_path}...")
        df = pd.read_csv(csv_path)

        if column_name not in df.columns:
            raise ValueError(f"La columna '{column_name}' no existe en el archivo CSV.")

        print(f"Processing column '{column_name}' for anonymization...")
        anonymized_column = []

        for text in df[column_name]:
            doc = nlp(str(text))
            anonymized_text = anonymize_text_with_regex(text, doc)
            anonymized_column.append(anonymized_text)

        # Agregar la columna anonimizada al DataFrame
        df[f"{column_name}_anonimizada"] = anonymized_column

        # Guardar el nuevo CSV
        df.to_csv(output_csv_path, index=False)
        print(f"CSV file with anonymized column generated at: {output_csv_path}")

        # Generar el archivo HTML para la columna anonimizada
        generate_html_anonymized_column(anonymized_column, output_html_path)

    except Exception as e:
        print(f"Error: {str(e)}")

if __name__ == "__main__":
    model_path = "./model-best"
    csv_path = "/home/sotavento/Documents/tejer_red/original_data/cedulassample.csv"
    column_name = "descripcion_desaparicion"  # Nombre de la columna a anonimizar
    output_csv_path = "./anonimized_output.csv"
    output_html_path = "./anonimized_column.html"

    process_and_anonymize_column(model_path, csv_path, column_name, output_csv_path, output_html_path)

In [ ]:
import presidio_anonymizer

anonymizer = presidio_anonymizer.Anonymizer()

# Define the text to be anonymized
text = "John Smith lives in New York and his phone number is (555) 123-4567."

# Anonymize the text using Presidio
anonymized_text = anonymizer.anonymize(text=text, analyzer=presidio_anonymizer.NlpEngine.ENGLISH)

# Print the anonymized text
print(anonymized_text)

In [ ]:
# download presidio
!pip install presidio_analyzer presidio_anonymizer
!python -m spacy download es_core_news_lg

In [ ]:
text_to_anonymize = "His name is Mr. Jones and his phone number is 212-555-5555"

In [ ]:
# download presidio
!pip install presidio_analyzer presidio_anonymizer
#!python -m spacy download es_core_news_lg
!python -m spacy download es_core_news_md

from presidio_analyzer import AnalyzerEngine, PatternRecognizer
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
import json
from pprint import pprint




In [ ]:
from presidio_analyzer import AnalyzerEngine, RecognizerRegistry
from presidio_analyzer.predefined_recognizers import EmailRecognizer
from presidio_analyzer.nlp_engine import NlpEngineProvider

LANGUAGES_CONFIG_FILE = "./languages-config.yml"

# Crear el motor NLP basado en el archivo de configuración
provider = NlpEngineProvider(conf_file=LANGUAGES_CONFIG_FILE)
nlp_engine_with_spanish = provider.create_engine()

# Configurar un reconocedor de correos electrónicos en inglés
email_recognizer_en = EmailRecognizer(supported_language="en", context=["email", "mail"])

# Configurar un reconocedor de correos electrónicos en español
email_recognizer_es = EmailRecognizer(supported_language="es", context=["correo", "electrónico"])

# Crear el registro de reconocedores
registry = RecognizerRegistry()

# Agregar los reconocedores al registro
registry.add_recognizer(email_recognizer_en)
registry.add_recognizer(email_recognizer_es)

# Configurar los idiomas soportados por el registro
registry.supported_languages = ["en", "es"]

# Configurar el motor de análisis con el registro actualizado
analyzer = AnalyzerEngine(
    registry=registry,
    supported_languages=["en", "es"],
    nlp_engine=nlp_engine_with_spanish
)

# Analizar un texto en inglés
results = analyzer.analyze(text="My name is David and my email is david@example.com", language="en")

# Imprimir los resultados del análisis
print("Resultados del análisis en inglés:")
for result in results:
    print(result)

# Analizar un texto en español
results_es = analyzer.analyze(text="Mi correo electrónico es david@ejemplo.com", language="es")

# Imprimir los resultados del análisis en español
print("\nResultados del análisis en español:")
for result in results_es:
    print(result)

    # Anonimizar los textos utilizando el motor de anonimización
    anonymized_results_en = anonymizer.anonymize(
        text="My name is David and my email is david@example.com",
        analyzer_results=results,
        operators={"EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "[EMAIL]"})}
    )

    anonymized_results_es = anonymizer.anonymize(
        text="Mi correo electrónico es david@ejemplo.com",
        analyzer_results=results_es,
        operators={"EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "[CORREO]"})}
    )

    # Presentar los textos anonimizados
    print("\nTexto anonimizado en inglés:")
    print(anonymized_results_en.text)

    print("\nTexto anonimizado en español:")
    print(anonymized_results_es.text)

In [ ]:
---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
Cell In[28], line 65
     55 text_to_anonymize = (
     56     "REFIERE LA REPORTARTE QUE EL DÍA 26 DE MAYO DEL 2024 APROXIMADAMENTE A LAS 15:30 "
     57     "SE ENCONTRABA EN EL DOMICILIO UBICADO EN SIERRA CHAPULTEPEC #174 COLONIA CHULA VISTA "
   (...)     61     "Y FAMILIAR OBTENIENDO RESULTADOS NEGATIVOS, DESDE ESE MOMENTO SE DESCONOCE DEL PARADERO."
     62 )
     64 # Analizar el texto para detectar entidades
---> 65 analyzer_results = analyzer.analyze(
     66     text=text_to_anonymize,
     67     entities=["PHONE_NUMBER", "ADDRESS", "PERSON"],
     68     language="es"
     69 )
     71 # Imprimir los resultados del análisis
     72 print("Resultados del análisis:")

File ~/Documents/tejer_red/.venv/lib/python3.12/site-packages/presidio_analyzer/analyzer_engine.py:206, in AnalyzerEngine.analyze(self, text, language, entities, correlation_id, score_threshold, return_decision_process, ad_hoc_recognizers, context, allow_list, allow_list_match, regex_flags, nlp_artifacts)
    163 """
    164 Find PII entities in text using different PII recognizers for a given language.
    165 
   (...)    201 
    202 """  # noqa: E501
    204 all_fields = not entities
--> 206 recognizers = self.registry.get_recognizers(
    207     language=language,
    208     entities=entities,
    209     all_fields=all_fields,
    210     ad_hoc_recognizers=ad_hoc_recognizers,
    211 )
    213 if all_fields:
    214     # Since all_fields=True, list all entities by iterating
    215     # over all recognizers
    216     entities = self.get_supported_entities(language=language)

File ~/Documents/tejer_red/.venv/lib/python3.12/site-packages/presidio_analyzer/recognizer_registry/recognizer_registry.py:197, in RecognizerRegistry.get_recognizers(self, language, entities, all_fields, ad_hoc_recognizers)
    191 logger.debug(
    192     "Returning a total of %s recognizers",
    193     str(len(to_return)),
    194 )
    196 if not to_return:
--> 197     raise ValueError("No matching recognizers were found to serve the request.")
    199 return list(to_return)

ValueError: No matching recognizers were found to serve the request.

In [ ]:
text_to_anonymize = "REFIERE LA REPORTARTE QUE EL DÍA 26 DE MAYO DEL 2024 APROXIMADAMENTE A LAS 15:30 SE ENCONTRABA EN EL DOMICILIO UBICADO EN SIERRA CHAPULTEPEC #174 COLONIA CHULA VISTA EN EL MUNICIPIO DE TLAJOMULCO DE ZÚÑIGA, JALISCO, (DICHO POR SU SOBRINA ALEJANDRA 3318430151) POSTERIORMENTE SALE DEL DOMICILIO Y MENCIONA QUE IRA A VENDER ROPA AL CECYTEJ TÉCNICO DE CHULAVISTA (SIN MENCIONAR MAS DATOS) AL DARSE CUENTA QUE NO REGRESO SE COMENZÓ CON LA BÚSQUEDA EN EL ENTORNO SOCIAL Y FAMILIAR OBTENIENDO RESULTADOS NEGATIVOS, DESDE ESE MOMENTO SE DESCONOCE DEL PARADERO."
analyzer = AnalyzerEngine()
    languages_config=nlp_config,

analyzer_results = analyzer.analyze(text=text_to_anonymize, entities=["PERSON"], language='es')

print(analyzer_results)

In [ ]:
titles_recognizer = PatternRecognizer(supported_entity="TITLE",
                                      deny_list=["Mr.","Mrs.","Miss"])

pronoun_recognizer = PatternRecognizer(supported_entity="PRONOUN",
                                       deny_list=["he", "He", "his", "His", "she", "She", "hers", "Hers"])

analyzer.registry.add_recognizer(titles_recognizer)
analyzer.registry.add_recognizer(pronoun_recognizer)

analyzer_results = analyzer.analyze(text=text_to_anonymize,
                            entities=["TITLE", "PRONOUN"],
                            language="en")
print(analyzer_results)


In [ ]:
analyzer_results = analyzer.analyze(text=text_to_anonymize, language='es')

analyzer_results

In [ ]:
anonymizer = AnonymizerEngine()

anonymized_results = anonymizer.anonymize(
    text=text_to_anonymize,
    analyzer_results=analyzer_results,    
    operators={"DEFAULT": OperatorConfig("replace", {"new_value": ""}), 
                        "PHONE_NUMBER": OperatorConfig("mask", {"type": "mask", "masking_char" : "*", "chars_to_mask" : 12, "from_end" : True}),
                        "TITLE": OperatorConfig("redact", {})}
)

print(f"text: {anonymized_results.text}")
print("detailed response:")

pprint(json.loads(anonymized_results.to_json()))